
##Load processed datasets

In this step, we load the training and test datasets generated by the data ingestion notebook.These datasets are stored as serialized pickle files to ensure reproducibility and a clear separation between data ingestion and preprocessing.No transformations are applied at this stage.


In [1]:
# Imports + paths
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PROCESSED = Path("../data/processed")

with open(DATA_PROCESSED / "train.pkl", "rb") as f:
    train_df = pickle.load(f)

with open(DATA_PROCESSED / "test.pkl", "rb") as f:
    test_df = pickle.load(f)

print(train_df.shape, test_df.shape)

(574945, 341) (107, 336)



##Identify identifier and label columns

This step defines columns that should not be used as model features.Identifier columns describe rows, sequences, or subjects, while label columns represent target variables.These columns are excluded to prevent data leakage and ensure that only sensor-driven measurements are used as inputs.


In [2]:
ID_COLS = ["sequence_id","row_id","sequence_counter", "subject", "phase"]
LABEL_COLS = ["gesture", "behavior"]

NON_FEATURE_COLS = set(ID_COLS + LABEL_COLS)

# Select feature columns
train_feature_cols = [c for c in train_df.columns if c not in NON_FEATURE_COLS]
test_feature_cols  = [c for c in test_df.columns  if c not in NON_FEATURE_COLS]

feature_cols = sorted(list(set(train_feature_cols).intersection(test_feature_cols)))

print("Train feature cols:", len(train_feature_cols))
print("Test feature cols :", len(test_feature_cols))
print("Common feature cols:", len(feature_cols))

Train feature cols: 334
Test feature cols : 332
Common feature cols: 332



##Filter short sequences

Sequences with fewer than a minimum number of timesteps are removed from the training dataset. This ensures that all retained sequences provide sufficient temporal context for sequence-based models such as CNNs, LSTMs, and BiLSTMs. The test dataset is left unchanged.


In [3]:
MIN_LEN = 35

seq_len = train_df.groupby("sequence_id").size()
keep_ids = seq_len[seq_len >= MIN_LEN].index

train_df = train_df[train_df["sequence_id"].isin(keep_ids)].copy()


##Save cleaned datasets

After preprocessing, the cleaned and normalized training and test datasets are serialized and saved as separate artifacts.These files serve as standardized inputs for downstream sequence construction and model training notebooks, ensuring reproducibility and modularity of the pipeline.


In [4]:
with open(DATA_PROCESSED / "train_clean.pkl", "wb") as f:
    pickle.dump(train_df, f)

with open(DATA_PROCESSED / "test_clean.pkl", "wb") as f:
    pickle.dump(test_df, f)

print("Saved cleaned datasets")

Saved cleaned datasets
